# Prompt

The `prompt.py` module defines `PromptTemplate`, a string-based prompt template used to create formatted prompts for language models. It supports `f-string`, Mustache, and Jinja2 template formats.

> **Security:** Prefer `f-string` templates when the template source is untrusted. Jinja2 sandboxing is a best-effort security measure and should not be treated as a complete security guarantee.

# PromptTemplate: `StringPromptTemplate`

`PromptTemplate` represents a reusable string template containing variables that are replaced with user-provided or partial values during formatting.

## Fields

1. `template`:`str`:= Stores the prompt template string.

2. `template_format`:`PromptTemplateFormat`:= Specifies the syntax used by the prompt template. Its default value is `"f-string"`. Supported values are `"f-string"`, `"mustache"`, and `"jinja2"`.

3. `validate_template`:`bool`:= Determines whether the template is validated during initialisation. Its default value is `False`.

## Properties

1. `lc_attributes`:`dict[str, Any]`:= Returns additional attributes required to serialize the prompt template.

   **Syntax**

   ```python
   @property
   def lc_attributes(
       self # Prompt template instance
   ) -> dict[str, Any]
   ```

2. `_prompt_type`:`str`:= Returns the serialization type key assigned to the prompt template.

   **Syntax**

   ```python
   @property
   def _prompt_type(
       self # Prompt template instance
   ) -> str
   ```

## Validators

1. `pre_init_validation`:= Validates the template configuration before model initialisation and determines the required input variables.

   Mustache templates cannot be validated using `validate_template=True`. When validation is enabled, `input_variables` must be supplied. Variables already provided through `partial_variables` are excluded from `input_variables`.

   This validator runs automatically before the prompt template is initialised.

   **Syntax**

   ```python
   @classmethod
   def pre_init_validation(
       cls, # PromptTemplate class
       values: dict[str, Any] # Values supplied during initialisation
   ) -> Any
   ```

## Methods

1. `get_lc_namespace`:= Returns the LangChain serialization namespace assigned to `PromptTemplate`.

   **Syntax**

   ```python
   @classmethod
   def get_lc_namespace(
       cls # PromptTemplate class
   ) -> list[str]
   ```

2. `get_input_schema`:= Returns the Pydantic model used to validate prompt input.

   Mustache templates generate the schema from their template structure. Other template formats use the inherited input-schema behaviour.

   **Syntax**

   ```python
   def get_input_schema(
       self, # Prompt template instance
       config: RunnableConfig | None = None # Runnable configuration
   ) -> type[BaseModel]
   ```

3. `__add__`:= Combines the current prompt with another `PromptTemplate` or string and returns a new `PromptTemplate`.

   Both prompt templates must use the same template format. A partial variable cannot be defined in both templates. Unsupported operand types raise `NotImplementedError`.

   **Syntax**

   ```python
   def __add__(
       self, # Current prompt template instance
       other: Any # PromptTemplate or string to append
   ) -> PromptTemplate
   ```

4. `format`:= Replaces template variables with supplied values and returns the formatted prompt string.

   **Syntax**

   ```python
   def format(
       self, # Prompt template instance
       **kwargs: Any # Values used to format the prompt
   ) -> str
   ```

5. `from_examples`:= Creates a prompt template by joining an optional prefix, a list of examples, and a suffix.

   **Syntax**

   ```python
   @classmethod
   def from_examples(
       cls, # PromptTemplate class
       examples: list[str], # Examples to include in the prompt
       suffix: str, # Text placed after the examples
       input_variables: list[str], # Variables expected by the final prompt
       example_separator: str = "\n\n", # Separator placed between prompt sections
       prefix: str = "", # Text placed before the examples
       **kwargs: Any # Additional PromptTemplate arguments
   ) -> PromptTemplate
   ```

6. `from_file`:= Loads a template string from a file and creates a `PromptTemplate`.

   **Syntax**

   ```python
   @classmethod
   def from_file(
       cls, # PromptTemplate class
       template_file: str | Path, # Path of the template file
       encoding: str | None = None, # Text encoding used to read the file
       **kwargs: Any # Additional PromptTemplate arguments
   ) -> PromptTemplate
   ```

7. `from_template`:= Creates a `PromptTemplate` directly from a template string.

   Input variables are detected automatically from the template. Variables supplied through `partial_variables` are removed from the required input variables.

   **Syntax**

   ```python
   @classmethod
   def from_template(
       cls, # PromptTemplate class
       template: str, # Template string to load
       *,
       template_format: PromptTemplateFormat = "f-string", # Template syntax
       partial_variables: dict[str, Any] | None = None, # Variables filled in advance
       **kwargs: Any # Additional PromptTemplate arguments
   ) -> PromptTemplate
   ```


In [1]:
from langchain_core.prompts import PromptTemplate # Import the PromptTemplate class

prompt = PromptTemplate.from_template( # Create a prompt directly from a template string
    "Translate {text} from {source_language} to {target_language}.", # Define the reusable template
    partial_variables={"source_language": "English"} # Fill the source language in advance
) # Finish creating the prompt template

print(prompt.input_variables) # Display the remaining required variables

result = prompt.format( # Format the prompt with the required values
    text="Good morning", # Supply the text to translate
    target_language="French" # Supply the target language
) # Finish formatting the prompt

print(result) # Display the formatted prompt

combined_prompt = prompt + "\nReturn only the translation." # Append an instruction to the prompt
print(combined_prompt.format(text="Thank you", target_language="Spanish")) # Format the combined prompt

input_schema = prompt.get_input_schema() # Generate the Pydantic input-validation model
print(input_schema.model_json_schema()) # Display the generated input schema

print(prompt.get_lc_namespace()) # Display the LangChain serialization namespace
print(prompt.lc_attributes) # Display the attributes used during serialization

['target_language', 'text']
Translate Good morning from English to French.
Translate Thank you from English to Spanish.
Return only the translation.
{'properties': {'target_language': {'title': 'Target Language', 'type': 'string'}, 'text': {'title': 'Text', 'type': 'string'}}, 'required': ['target_language', 'text'], 'title': 'PromptInput', 'type': 'object'}
['langchain', 'prompts', 'prompt']
{'template_format': 'f-string'}
